# Capstone build --- Chapter 13: Escalation

A governed agent must know when not to act. The complaint agent escalates to a human on three paths: when a prior tool call was denied by a gate, when the regulatory check flags a risk it must not auto-resolve, and when a claim would reach the draft without evidence. Chapter~13 drives cases down these paths and shows the human-review interface the escalation hands off to. It runs the real harness.

In [ ]:
import json
from pathlib import Path
from agentlab.capstone import build_complaint_harness
from agentlab.core import Budget, BudgetTracker, TaskSpec

root = next((c for c in (Path('.'), Path('..'), Path('../code'), Path('code'))
             if (c / 'data' / 'eval_cases' / 'cases.json').exists()), Path('.'))
cases = json.loads((root / 'data' / 'eval_cases' / 'cases.json').read_text())
harness, registry = build_complaint_harness(policies_dir=root / 'data' / 'policies')

def run(case_id):
    case = next(c for c in cases if c['id'] == case_id)
    task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
    traj = harness.run(task, max_steps=16, budget_tracker=BudgetTracker(Budget(tool_calls=20)))
    return case, traj

## Escalation on regulatory risk

When `flag_regulatory` marks a UDAAP or Reg X risk, the agent does not draft a reply: it raises an `Escalate` naming the flags. The escalation is a step in the trajectory, so its reason is recorded and auditable.

In [ ]:
case, traj = run('case-016')
print('message:', case['message'])
print('status :', traj.final_state.status)
esc = next((r for r in traj.records if r.action.kind == 'escalate'), None)
print('reason :', esc.action.reason if esc else '(no escalation)')

## Escalation on a denied tool call

A message carrying PII is denied at the policy gate, and the agent reads that failed result and escalates rather than proceeding. This is the same failed-result-to-escalation path the plan of Chapter~8 diverted on, now driven by a real gate refusal.

In [ ]:
case, traj = run('case-011')
print('message:', case['message'])
print('status :', traj.final_state.status)
failed = next((r for r in traj.records if r.action.kind == 'tool_call'
               and r.observation and not r.observation.get('success')), None)
if failed:
    print('denied :', failed.observation['error'])
esc = next((r for r in traj.records if r.action.kind == 'escalate'), None)
print('reason :', esc.action.reason if esc else '(no escalation)')

## The human-review interface

An escalation is a hand-off, not a dead end. `harness.run` accepts a `human_reviewer`; an `EscalationRequest` carries the run id, the step, the reason and the gate results to the reviewer, who returns a decision. `ScriptedReviewer` supplies fixed decisions, which is what a test uses in place of a person at the console.

In [ ]:
from agentlab.governance.escalation import (
    ScriptedReviewer, HumanResponse, HumanDecision,
)
reviewer = ScriptedReviewer([HumanResponse(decision=HumanDecision.DEFER, note='needs analyst review')])
case, _ = run('case-016')
task = TaskSpec(goal='handle complaint', inputs={'message': case['message']})
traj = harness.run(task, max_steps=16,
                   budget_tracker=BudgetTracker(Budget(tool_calls=20)),
                   human_reviewer=reviewer)
print('final status with reviewer:', traj.final_state.status)

Escalation is how the agent stays inside its authority: a denied call, a flagged regulation or an ungrounded claim all route to a human with the context of the decision, recorded in the audit chain. Chapter~14 generalizes the single agent into a supervised set of workers, and Chapter~15 assembles the whole capstone.